# Step 6 — get the parallel-placement Bottleneck checkpoint (Colab)

Trains **one** config, `exp_phase3_placement_parallel_evidential` — the best result in
the whole thesis (0.915 accuracy, beats even Full-FT's 0.905, at ~0.3% of its
parameters; also the best OOD-detection numbers of any run in Steps 4.5-7). This is
the exact checkpoint shape `app/backend/ml/adapter.py` in the Sentinel demo app
already knows how to load (Step-6 in-block placement, serial/parallel forward-hook
form).

**Why Colab and not the Kaggle notebook (`notebooks/step6_placement.ipynb`):** that
run's checkpoint got lost because nobody clicked "Save Version" before the Kaggle
session ended -- Kaggle's `/kaggle/working` is ephemeral unless you explicitly commit
it. This notebook clones the repo **into Google Drive** instead, so everything
written under the repo (checkpoints included) is durable the moment it's written --
there's no separate save step to forget.

**Expected time:** `step6_placement.ipynb` states ~30-60 min for all 4 placement
configs on a Kaggle T4. Running just this one config should be roughly a quarter of
that -- ballpark **10-20 min** total (train + 600-episode eval) on a Colab T4.

Run cells top to bottom.

## 0. GPU check

In [ ]:
import torch
print('torch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
assert torch.cuda.is_available(), 'Enable a GPU: Runtime > Change runtime type > GPU (T4 is fine)'


## 1. Mount Drive + clone the repo

Clones into `MyDrive/bpeft_step6/thesis` (not `/content`), so the clone -- and
everything scripts/train.py writes under it, including `checkpoints/` -- survives a
runtime disconnect/restart. Branch is `main`: Step 5/6/7 are already merged there
(verified against `origin/main` before writing this notebook).

In [ ]:
import os, sys, subprocess, shutil
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
print("In Colab:", IN_COLAB)

GITHUB_URL = "https://github.com/notAvailable73/thesis"
BRANCH = "main"

def _looks_like_repo(p: Path) -> bool:
    return (p / "src").is_dir() and (p / "configs").is_dir()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    WORKDIR = "/content/drive/MyDrive/bpeft_step6"
    os.makedirs(WORKDIR, exist_ok=True)
    REPO = Path(WORKDIR) / "thesis"

    def _cur_branch(p):
        r = subprocess.run(["git", "-C", str(p), "rev-parse", "--abbrev-ref", "HEAD"],
                            capture_output=True, text=True)
        return r.stdout.strip()

    def _clone():
        print(f"Cloning {GITHUB_URL} (branch {BRANCH}) into {REPO} ...")
        subprocess.run(["git", "clone", "--branch", BRANCH, GITHUB_URL, str(REPO)], check=True)

    if not _looks_like_repo(REPO):
        _clone()
    elif _cur_branch(REPO) != BRANCH:
        print(f"Existing clone on '{_cur_branch(REPO)}', need '{BRANCH}' -- re-cloning ...")
        shutil.rmtree(str(REPO))
        _clone()
    else:
        print(f"Reusing clone at {REPO} (branch {BRANCH}), pulling ...")
        subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)
else:
    cur = Path.cwd().resolve()
    REPO = next((c for c in [cur, *cur.parents] if _looks_like_repo(c)), None)
    if REPO is None:
        raise RuntimeError("repo root not found (no src/ + configs/ above CWD)")

os.chdir(REPO)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))
print("Repo root:", REPO)
_h = subprocess.run(["git", "-C", str(REPO), "log", "--oneline", "-1"],
                     capture_output=True, text=True).stdout.strip()
print(">>> RUNNING CODE AT:", _h)
assert (REPO / "configs" / "exp_phase3_placement_parallel_evidential.yaml").exists(), \
    "Step 6 config missing -- wrong branch cloned?"


## 2. Install dependencies

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("deps installed")


## 3. Build the frozen CIFAR-FS split

`data/` is gitignored; this materializes the canonical 64/16/20 Bertinetto split.
Do NOT hand-edit the JSON it writes.

In [ ]:
import subprocess
subprocess.run([sys.executable, "scripts/build_cifar_fs_split.py"], check=True)
import json
sp = json.load(open("data/cifar_fs_split.json"))
print("split status:", sp.get("_status"))
print("sizes:", {k: len(v) for k, v in sp.items() if isinstance(v, list)})


## 4. Stage CIFAR-100 from Drive (avoids the slow cs.toronto.edu host)

`cs.toronto.edu` serves `cifar-100-python.tar.gz` at ~7 KB/s to Colab -- a run can
stall ~45 min and then fail. If you (or a teammate) already have the tarball staged
at `MyDrive/bpeft_data/cifar-100-python.tar.gz` from a previous Step 4/5 run, this
cell reuses it instantly. If not, it falls back to a direct download with a loud
warning -- let it run, or grab the tarball from this repo's local `data/` folder and
upload it to that Drive path first, then re-run this cell.

md5 of the canonical tarball: `eb9058c3a382ffc7106e4002c42a8d85`.

In [ ]:
import os, shutil, hashlib

DRIVE_TARBALL = "/content/drive/MyDrive/bpeft_data/cifar-100-python.tar.gz"
EXPECT_MD5 = "eb9058c3a382ffc7106e4002c42a8d85"
dst = "data/cifar-100-python.tar.gz"

os.makedirs("data", exist_ok=True)
if os.path.exists(dst) and os.path.getsize(dst) > 0:
    print(f"[cifar] already present at {dst} - nothing to do")
elif os.path.exists(DRIVE_TARBALL):
    shutil.copy(DRIVE_TARBALL, dst)
    md5 = hashlib.md5(open(dst, "rb").read()).hexdigest()
    assert md5 == EXPECT_MD5, f"md5 mismatch: got {md5}, expected {EXPECT_MD5}"
    print(f"[cifar] staged {dst} from Drive (md5 OK, {os.path.getsize(dst)/1e6:.0f} MB)")
else:
    print(f"[cifar] WARNING: {DRIVE_TARBALL} not found in Drive.")
    print("        Falling back to the direct cs.toronto.edu download -- this can")
    print("        stall for ~45 min before failing. Consider Ctrl+C-ing this cell,")
    print("        uploading data/cifar-100-python.tar.gz from your local clone to")
    print(f"        {DRIVE_TARBALL}, and re-running.")
    # No action needed here -- scripts/build_cifar_fs_split.py / torchvision will
    # attempt the live download the next time CIFAR-100 is touched.


## 5. Stage SVHN (evaluate.py always builds a far-OOD pool from it)

Unlike CIFAR-100's Toronto host, SVHN's `ufldl.stanford.edu` host has been reliably
fast from cloud notebooks, so this just lets it download live. If that ever proves
flaky, stage `test_32x32.mat` (md5 `eb5a983be6a315427106f1b164d9cef3` (verified against this repo's local copy)) to
`MyDrive/bpeft_data/test_32x32.mat` the same way as CIFAR-100 above and this cell
will pick it up instead.

In [ ]:
import os, shutil

DRIVE_SVHN = "/content/drive/MyDrive/bpeft_data/test_32x32.mat"
dst = "data/svhn/test_32x32.mat"
os.makedirs("data/svhn", exist_ok=True)

if os.path.exists(dst) and os.path.getsize(dst) > 0:
    print(f"[svhn] already present at {dst} - nothing to do")
elif os.path.exists(DRIVE_SVHN):
    shutil.copy(DRIVE_SVHN, dst)
    print(f"[svhn] staged {dst} from Drive ({os.path.getsize(dst)/1e6:.0f} MB)")
else:
    print("[svhn] not staged -- will download live from ufldl.stanford.edu on first use")


## 6. Train + evaluate

`RUN_SOFTMAX_TOO = False` by default -- the Sentinel app only needs the evidential
head (that's the whole "honest AI" uncertainty story), so skip the softmax run to
save time unless you want it for your own comparison table.

`RUN_EVAL = True` runs the full 600-episode evaluation after training, which is what
lets you sanity-check the result against the thesis's reported 0.915 accuracy /
0.933 far-OOD AUROC. Training alone already writes the checkpoint if you're in a
hurry and want to skip straight to Section 8 -- set this to `False` to shave a few
minutes.

In [ ]:
import subprocess, os

RUN_SOFTMAX_TOO = False
RUN_EVAL = True
NUM_EPISODES = 600

RUNS = [("exp_phase3_placement_parallel_evidential", "evidential")]
if RUN_SOFTMAX_TOO:
    RUNS.append(("exp_phase3_placement_parallel_softmax", "softmax"))

def result_path(interp):
    return f"results/phase3_placement_parallel_bottleneck_prototype-{interp}_metrics.json"

def run(cmd):
    print(">>>", " ".join(cmd), flush=True)
    return subprocess.run(cmd).returncode

status = {}
for name, interp in RUNS:
    cfg = f"configs/{name}.yaml"
    print(f"\n{'='*72}\n== {name} ==\n{'='*72}", flush=True)
    rc = run([sys.executable, "scripts/train.py", "--config", cfg, "--wandb-mode", "disabled"])
    if rc != 0:
        status[name] = f"TRAIN failed (rc={rc})"
        continue
    if not RUN_EVAL:
        status[name] = "TRAIN OK (eval skipped)"
        continue
    out = result_path(interp)
    eval_cmd = [sys.executable, "scripts/evaluate.py", "--config", cfg,
                "--num-episodes", str(NUM_EPISODES), "--wandb-mode", "disabled",
                "--results-suffix", "phase3_placement_parallel"]
    rc = run(eval_cmd)
    status[name] = "OK" if (rc == 0 and os.path.exists(out)) else f"EVAL failed (rc={rc})"

print("\n" + "=" * 40 + "\nRUN STATUS\n" + "=" * 40)
for name, _ in RUNS:
    print(f"  {name:44s} {status.get(name, 'not run')}")


## 7. Results summary (sanity-check against the thesis's reported numbers)

In [ ]:
import json, glob

for f in sorted(glob.glob("results/phase3_placement_parallel_*_metrics.json")):
    d = json.load(open(f))
    print(f)
    print("  accuracy_mean               :", d.get("accuracy_mean"))
    print("  ece_pooled                  :", d.get("ece_pooled"))
    print("  best_val_epoch              :", d.get("best_val_epoch"))
    print("  ood_auroc__svhn_far         :", d.get("ood_auroc__svhn_far__vacuity"))
    print("  ood_auroc__cifar100_near    :", d.get("ood_auroc__cifar100_near__vacuity"))
    print("  n_params                    :", d.get("n_params"))

print()
print("Expected from step_writeups/step6.txt (parallel, evidential): accuracy 0.915,")
print("far-OOD SVHN AUROC 0.933, near-OOD CIFAR-100 AUROC 0.912. Small deviations (a")
print("few thousandths) are normal run-to-run noise; a large gap means something's off.")


## 8. Get the checkpoint

Because the whole repo clone lives on Drive, the checkpoint is **already durable**
the moment `scripts/train.py` finished writing it in Section 6 -- there's nothing
extra to click or commit, unlike the Kaggle notebooks. This cell just locates it,
confirms it matches the app's expected filename, and (in Colab) triggers a direct
browser download so you can drop it straight into this repo's local `checkpoints/`
folder.

In [ ]:
import glob

matches = sorted(glob.glob("checkpoints/model_phase2_bottleneck_prototype-evidential_seed*.pt"))
assert matches, "No checkpoint found -- did Section 6's training step succeed?"
ckpt_path = matches[-1]
print("Checkpoint:", ckpt_path)
print("Size: {:.1f} MB".format(os.path.getsize(ckpt_path) / 1e6))
print("Persisted at (Drive path):", os.path.abspath(ckpt_path))
print()
print("Next step: copy this file into your LOCAL repo clone at the same relative")
print(f"path -- checkpoints/{os.path.basename(ckpt_path)} -- no SENTINEL_CHECKPOINT_PATH")
print("override needed, this is already the app's default filename.")

if IN_COLAB:
    from google.colab import files
    files.download(ckpt_path)
